# 第8日 03：シーケンス制御 / Day 8 03: Sequence Control

**所要時間：135分 / Duration: 135 minutes.** [第7〜9日 時間割 / Day 7–9 syllabus](../../COURSE_SYLLABUS_DAY07_09.md)

機械は「同時に全部動く」のではなく、決まった順序で状態を変えます。この教材では、状態・遷移条件・インターロック・故障対応・HMIを、連続シート検査機の架空モデルで学びます。
Machines do not start everything at once; they change state in a defined order. This lesson covers states, transition conditions, interlocks, fault handling and the HMI using the fictional continuous-sheet inspection machine.

**到達目標 / Objectives:** 状態遷移図を読み書きし、プロセス／品質／安全のインターロックを区別し、故障とNGを分けて対応を説明できる。
Read and write state diagrams, distinguish process, quality and safety interlocks, and separate a machine fault from an NG result when describing a response.



## 1. なぜ順序が必要か / Why sequences are needed

材料がないのに検査を始めれば、カメラは空の背景を撮ります。停止処理を飛ばして電源を切れば、材料がたるみ、次回の開始でトラブルになります。**順序**は、機械と材料と人の安全を守るための約束です。

If inspection starts with no material, the camera images an empty background. If stopping is skipped, the web slackens and the next start causes trouble. **Order** is the agreement that protects the machine, the material and people.

代表的な状態は **待機（Idle）・準備完了（Ready）・起動中（Starting）・運転中（Running）・検査中（Inspecting）・停止処理（Stopping）・故障（Fault）・復旧（Recovery）** です。すべての状態に「入る理由」と「出る条件」があります。

Typical states are **Idle, Ready, Starting, Running, Inspecting, Stopping, Fault and Recovery**. Every state has a reason to enter and a condition to leave.



## 2. 状態機械 / State machines

**状態機械（state machine）**は、状態と遷移の集まりです。遷移には必ず**条件**があります。条件のない遷移は、設計の抜けです。たとえば「待機から準備完了へ」は「コントローラ正常・センサ正常・カメラ準備完了・AI正常・未解決の品質対応なし・許可されたモード」がそろったとき、と定義します。

A **state machine** is a set of states and transitions. Every transition has a **condition**; a transition without one is a design gap. For example, Idle → Ready might require: controller healthy, sensors healthy, camera ready, AI healthy, no unresolved quality action and an authorised mode.

![機械の状態遷移 / Machine-state sequence](../../assets/diagrams/day08/D08-10.svg)

**図 D08-10 / Figure D08-10 — 確認 / check:** 「故障」から「運転中」へ直接戻ってよいですか？ / May the machine return directly from Fault to Running?

**これらの準備条件は、認証された安全条件とは別です。**「AI正常」は安全の証明ではありません。安全条件は、独立した安全系が判断します。
These readiness conditions are **not certified safety conditions**. “AI healthy” does not prove safety; independent safety systems decide safety conditions.



## 3. 起動と停止 / Start and stop conditions

**起動要求**は、すべての準備条件が満たされたときに受け付けます。**停止要求**は、いつでも受け付け、状態を安全側へ移します。**制御停止（controlled stop）**と**緊急停止（emergency stop）**は別の概念です。緊急停止は安全系の機能であり、通常のラダーで代用できません。

A **start request** is accepted only when all readiness conditions hold. A **stop request** is accepted at any time and moves the machine towards a safe state. A **controlled stop** and an **emergency stop** are different: the emergency stop is a safety-system function and cannot be replaced by ordinary ladder logic.

**故障後の自動再起動**は危険な場合があります。原因が除去され、人が再起動を許可する設計を検討します。「リセットを押したら勝手に動き出す」設計は避けます。
**Automatic restart after a fault** can be hazardous. Consider a design where the cause is cleared and a person authorises restart. Avoid “press reset and it starts moving by itself”.



## 4. インターロックの3種類 / Three classes of interlock

1. **プロセスインターロック**：材料がない、速度が範囲外など、工程を守る条件。
2. **品質インターロック**：AI結果が無効、閾値未満など、品質判断を守る条件。
3. **安全関連インターロック**：ガード開放、非常停止など、人と機械の危害を防ぐ条件。

1. **Process interlock:** protects the process, e.g. no material or speed out of range.
2. **Quality interlock:** protects the quality decision, e.g. invalid AI result or score below threshold.
3. **Safety-related interlock:** prevents harm, e.g. guard open or emergency stop.

**この3つを混ぜると危険です。**「AIがNGを出したから停止」は品質対応であり、安全機能ではありません。安全機能は、リスクアセスメントに基づき、別の機器・別の設計・別の検証で実現します。
**Mixing these is dangerous.** “Stop because AI said NG” is a quality response, not a safety function. Safety functions use separate devices, design and validation based on a risk assessment.

![プロセスインターロックと安全機能 / Process interlocks versus safety functions](../../assets/diagrams/day08/D08-12.svg)

**図 D08-12 / Figure D08-12 — 確認 / check:** 境界線の左と右で、設計・検証・責任はどう変わりますか？ / How do design, validation and responsibility differ across the boundary?



## 5. タイミング / Timing

シーケンスには時間の要素があります。**時間遅延**（決まった時間待つ）、**タイムアウト**（決まった時間内に起きなければ異常）、**イベント順序**（Aの後にB）、**センサの応答**、**イベントの欠落**、**工程の待ち**です。

Sequences involve time: **time delay** (wait a defined period), **timeout** (an event must occur within a period), **event order** (B after A), **sensor response**, **missing events** and **process waiting**.

例えば「起動指令の後、2秒以内に材料有無センサがONにならなければ故障」と決めます。この「2秒」は仮定値であり、実機では材料・速度・機構に合わせて決めます。
For example: “within 2 s after the start command, the material-presence sensor must turn ON, otherwise fault.” The 2 s is hypothetical; a real machine sets it from material, speed and mechanics.



## 6. 故障対応 / Fault handling

故障の例を挙げます。**センサが応答しない**、**アクチュエータが所定位置に着かない**、**カメラが準備できない**、**PC通信が途絶える**、**AI結果がタイムアウトする**、**結果が無効・古い**。それぞれに、症状・原因候補・検出方法・記録すべき情報・人の確認が必要かを決めます。

Examples: **sensor not responding**, **actuator not reaching position**, **camera not ready**, **PC communication lost**, **AI result timeout**, **invalid or stale result**. For each, define symptoms, candidate causes, detection method, information to log and whether human review is required.

| 現象 / event | 検出の例 / detection | 品質対応の例 / quality response |
|---|---|---|
| カメラ断 / camera lost | 状態監視のタイムアウト / status timeout | 検査を保留 / hold inspection |
| 画像欠落 / missing image | 連番・エンコーダ間隔の不一致 / sequence gap | 該当区間を保留 / hold the segment |
| AI無効 / invalid result | 有効フラグ / validity flag | 良品にしない / never OK |
| 通信断 / comms lost | ハートビート途絶 / heartbeat lost | 保留・警報 / hold and alarm |
| 古いID / stale ID | ID照合 / ID comparison | 結果を破棄 / discard |

**機械の故障と検査のNGは別です。**故障は「機械が正しく動いていない」、NGは「製品が基準を満たさない」です。混同すると、故障中に作られた製品を良品扱いする危険があります。
**A machine fault and an inspection NG are different.** A fault means the machine is not working correctly; NG means the product does not meet criteria. Confusing them risks treating product made during a fault as good.

**必須の原則 / Mandatory rule:** 有効なAI結果がないまま、製品を自動的に「良品」にしてはいけません。
Without a valid AI result, the product must never automatically become “OK”.



## 7. HMI / HMI

**HMI（Human-Machine Interface）**は、操作と表示の窓口です。起動・停止の操作、状態表示、警報、エラーコード、作業者へのメッセージ、読み取り専用の診断、権限に応じたパラメータ変更を扱います。HMIは**安全条件を無効にする画面ではありません**。

An **HMI** is the window for operation and display: start/stop controls, status, alarms, error codes, operator messages, read-only diagnostics and permission-based parameter changes. The HMI is **not a screen that disables safety conditions**.

![HMI・PLC・診断情報 / HMI, PLC and diagnostics](../../assets/diagrams/day08/D08-11.svg)

**図 D08-11 / Figure D08-11 — 確認 / check:** 警報画面に最低限必要な情報は何ですか？ / What is the minimum information an alarm screen must show?

警報には「何が起きたか」「何を確認するか」「誰が対応するか」を含めます。警報を消すだけの操作（ acknowledge ）は、原因の除去ではありません。記録には、時刻・状態・操作者・対応内容を残します。
An alarm should state what happened, what to check and who responds. Acknowledging an alarm does not remove its cause. Records include time, state, operator and action.



## 8. シーケンスのシミュレーション / Simulated sequence

`day07_09_support/simulations/sim_machine_sequence.html` は、状態と遷移条件を操作できる**完全オフライン**の教材です。実機は動かしません。条件を切り替え、どの条件でどの状態へ移るかを観察します。

`day07_09_support/simulations/sim_machine_sequence.html` is a **fully offline** teaching tool for states and transitions. It never drives real equipment. Toggle conditions and observe which state is entered and why.

- 材料あり / material present
- カメラ準備完了 / camera ready
- AI正常 / AI healthy
- 起動要求 / start request
- 停止要求 / stop request
- 故障注入 / inject fault
- リセットと再起動許可 / reset and restart permit

**安全のための既定動作**：故障中は起動要求を受け付けず、リセット後も人が再起動を許可するまで運転しません。無効なAI結果は、良品として扱いません。
**Safe default behaviour:** while faulted, start requests are refused; after reset the machine waits for an explicit restart permit. Invalid AI results are never treated as OK.



## 9. 演習 / Exercises

1. 待機から準備完了へ移る条件を五つ書いてください。 / Write five conditions for Idle → Ready.
2. プロセス・品質・安全のインターロックを一つずつ挙げてください。 / Give one example of each interlock class.
3. 「材料有無センサが起動後2秒でONにならない」を、症状・検出・対応で整理してください。 / Organise “presence sensor not ON within 2 s of start” as symptom, detection, response.
4. 故障中に作られた製品を「良品」にしてはいけない理由を説明してください。 / Why must product made during a fault not be marked OK?
5. 停止要求と起動要求が同時に来たときの動作を設計してください。 / Design the behaviour when stop and start arrive together.

### 解答例 / Worked answers

1. 例：コントローラ正常、センサ正常、カメラ準備完了、AI正常、未解決の品質対応なし、許可モード。 / Examples: controller healthy, sensors healthy, camera ready, AI healthy, no unresolved quality action, authorised mode.
2. プロセス＝材料なしで開始しない、品質＝無効結果をOKにしない、安全＝ガード開放で危険動作を停止。 / Process: no start without material; quality: never OK an invalid result; safety: guard open stops hazardous motion.
3. 症状＝材料があるのに検出しない。検出＝起動後2秒のタイマで判定。対応＝故障状態へ移行、記録、人による確認。 / Symptom: material present but undetected; detection: 2 s timer; response: enter Fault, log, human review.
4. 故障中は検査が正しく行われた保証がないため。無効結果をOKにすると不良流出につながる。 / During a fault there is no assurance inspection was valid; treating invalid results as OK can release defects.
5. 停止を優先し、起動は受け付けない。再起動は人が許可する。 / Stop wins; start is refused; restart requires human authorisation.



## 10. まとめ / Summary

シーケンス制御は、状態と遷移条件の設計です。準備条件は安全条件ではありません。プロセス・品質・安全のインターロックを分け、故障とNGを区別し、無効結果をOKにしないことが核心です。HMIは操作と表示の窓口であり、安全を無効にする画面ではありません。

Sequence control is the design of states and transition conditions. Readiness is not safety. Separate process, quality and safety interlocks, distinguish fault from NG and never turn an invalid result into OK. The HMI is an interface for operation and display, not a way to disable safety.

**完了条件 / Completion criteria:** 5つの演習に答え、「PLC入出力・シーケンス表 / PLC I/O and Sequence Worksheet」の下書きを作れること。
**Completion criteria:** answer the five exercises and draft the **PLC I/O and Sequence Worksheet**.

**安全に関する注意 / Safety note:** 実際のシーケンス変更は、リスクアセスメント・検証・権限・現地手順のもとで行います。本教材のシミュレーションは実機を制御しません。
**Safety note:** Real sequence changes require risk assessment, verification, authority and site procedure. The simulations never control real equipment.
